# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import logging
import sys
import gc
from pathlib import Path

import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from src.utils.notebook_setup import init_nlp_notebook # noqa: E402

cfg = init_nlp_notebook()
cfg.paths.data_dir = str(project_root / "data")
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = instantiate(cfg.model.tokenizer).build()
print(f"Модель: {cfg.model.builder.model_name_or_path}")
print(f"Device: {device}")

NLP Environment ready. Root: c:\fake-news-detection-ml-system


c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: DeepPavlov/rubert-base-cased


Модель: DeepPavlov/rubert-base-cased
Device: cpu


# Утилита для быстрого сравнения конфигураций LoRA

In [ ]:
from peft import LoraConfig, get_peft_model # noqa: E402
import pandas as pd # noqa: E402

def build_lora_model(base_model, r, lora_alpha, target_modules, lora_dropout=0.05):
    """
    Оборачивает модель в LoRA с заданными параметрами.
    Возвращает модель и статистику параметров.
    """
    config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="SEQ_CLS",
    )
    peft_model = get_peft_model(base_model, config)
    trainable, total = peft_model.get_nb_trainable_parameters()
    return peft_model, {
        "r": r,
        "alpha": lora_alpha,
        "targets": str(target_modules),
        "trainable": trainable,
        "total": total,
        "trainable_%": round(100 * trainable / total, 4),
    }


def reset_model():
    """Пересобирает чистую базовую модель без LoRA."""
    builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
    # Временно отключаем peft чтобы получить чистую модель
    builder.finetuning_type = "full"
    builder.peft_config = None
    return builder.build()

# Эксперимент: сравниваем конфигурации LoRA

In [3]:
# Конфигурации которые хотим сравнить
# Для BERT: query и value — стандарт, можно добавить key и dense
experiments = [
    # (r, alpha, target_modules)
    (4,  8,  ["query", "value"]),
    (8,  16, ["query", "value"]),           # текущая конфигурация
    (16, 32, ["query", "value"]),
    (8,  16, ["query", "key", "value"]),
    (8,  16, ["query", "value", "dense"]),  # добавляем выходные проекции
]

results = []
for r, alpha, targets in experiments:
    base = reset_model()
    _, stats = build_lora_model(base, r, alpha, targets)
    results.append(stats)
    del base
    gc.collect()

df = pd.DataFrame(results)
print("=== Сравнение конфигураций LoRA ===\n")
print(df.to_string(index=False))

INFO:src.core.models.builder:Загрузка модели из: DeepPavlov/rubert-base-cased
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17522.13it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

=== Сравнение конфигураций LoRA ===

 r  alpha                     targets  trainable     total  trainable_%
 4      8          ['query', 'value']     148994 178003972       0.0837
 8     16          ['query', 'value']     296450 178151428       0.1664
16     32          ['query', 'value']     591362 178446340       0.3314
 8     16   ['query', 'key', 'value']     443906 178298884       0.2490
 8     16 ['query', 'value', 'dense']    1193474 179048452       0.6666


# Проверка что LoRA применилась к правильным слоям

In [ ]:
from peft import PeftModel # noqa: E402

# Берём текущий конфиг из yaml
base = reset_model()
model, stats = build_lora_model(
    base,
    r=cfg.model.builder.peft_config.r,
    lora_alpha=cfg.model.builder.peft_config.lora_alpha,
    target_modules=list(cfg.model.builder.peft_config.target_modules),
)

print(f"Trainable: {stats['trainable']:,} / {stats['total']:,} ({stats['trainable_%']}%)\n")

print("Слои с LoRA адаптерами (layer 0):")
for name, _ in model.named_modules():
    if "encoder.layer.0" in name and "lora" in name.lower():
        print(f"  {name}")

print("\nЗамороженные слои (первые 5):")
frozen = [n for n, p in model.named_parameters() if not p.requires_grad]
for n in frozen[:5]:
    print(f"  {n}")
print(f"  ... и ещё {len(frozen) - 5}")

INFO:src.core.models.builder:Загрузка модели из: DeepPavlov/rubert-base-cased
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17900.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

Trainable: 296,450 / 178,151,428 (0.1664%)

Слои с LoRA адаптерами (layer 0):
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_dropout
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_dropout.default
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_A
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_B
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_embedding_A
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_embedding_B
  base_model.model.bert.encoder.layer.0.attention.self.query.lora_magnitude_vector
  base_model.model.bert.encoder.layer.0.attention.self.value.lora_dropout
  base_model.model.bert.encoder.layer.0.attention.self.value.lora_dropout.default
  base_model.model.bert.encoder.layer.0.attention.self.value.lora_A
  base_model.mo

# Sanity Check

In [ ]:
from src.core.data.builder import NLPDataModule # noqa: E402

datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="validate")

batch = next(iter(datamodule.val_dataloader()))
batch_gpu = {k: v.to(device) for k, v in batch.items()}

# Оборачиваем в NLPModel чтобы проверить полный forward
from src.training.module import NLPModel # noqa: E402
nlp_model = NLPModel(model=model, optimizer_cfg=cfg.model_module.optimizer_cfg)
nlp_model.to(device)
nlp_model.eval()

with torch.no_grad():
    outputs = nlp_model(**batch_gpu)

print(f"Logits shape: {outputs.logits.shape}")  # [batch, 2]
print(f"Loss: {outputs.loss:.4f}")
print(f"Predictions: {torch.argmax(outputs.logits, dim=-1)[:8].tolist()}")
print(f"Labels:      {batch['labels'][:8].tolist()}")
print("\nFull forward pass через LoRA модель — OK")

INFO:src.core.data.builder:Нашли кэш данных: c:\fake-news-detection-ml-system\data/processed\nlp_dataset_cleaned_85974487. Очистка пропущена.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Logits shape: torch.Size([4, 2])
Loss: 0.7940
Predictions: [1, 1, 1, 1]
Labels:      [0, 0, 0, 0]

Full forward pass через LoRA модель — OK


# Эксперимент с lora_alpha / r ratio

In [ ]:
# Стандартное правило: alpha = 2*r или alpha = r
# alpha/r — это масштабирующий коэффициент обновлений
# Высокий alpha/r → модель быстрее меняется, риск нестабильности
# Низкий alpha/r → обновления мягче, обучение стабильнее

ratios = [
    (8, 8,  "alpha == r (консервативно)"),
    (8, 16, "alpha == 2r (стандарт)"),
    (8, 32, "alpha == 4r (агрессивно)"),
]

print("=== Влияние alpha/r ratio ===\n")
print(f"{'r':>4} {'alpha':>6} {'ratio':>6} {'описание'}")
print("-" * 50)
for r, alpha, desc in ratios:
    print(f"{r:>4} {alpha:>6} {alpha/r:>6.1f}  {desc}")

print("\nВывод: для спам-классификации рекомендуется alpha=2*r (стандарт).")
print(f"Текущий конфиг: r={cfg.model.builder.peft_config.r}, "
      f"alpha={cfg.model.builder.peft_config.lora_alpha}, "
      f"ratio={cfg.model.builder.peft_config.lora_alpha / cfg.model.builder.peft_config.r:.1f}")

=== Влияние alpha/r ratio ===

   r  alpha  ratio описание
--------------------------------------------------
   8      8    1.0  alpha == r (консервативно)
   8     16    2.0  alpha == 2r (стандарт)
   8     32    4.0  alpha == 4r (агрессивно)

Вывод: для спам-классификации рекомендуется alpha=2*r (стандарт).
Текущий конфиг: r=8, alpha=16, ratio=2.0


# Итоговая рекомендация

In [9]:
print("=== Рекомендуемая конфигурация для спам-классификации ===\n")

recommended = {
    "r": 8,
    "lora_alpha": 16,
    "target_modules": ["query", "value"],
    "lora_dropout": 0.05,
    "bias": "none",
}

print("configs/model/builder/default.yaml:")
print("  peft_config:")
for k, v in recommended.items():
    print(f"    {k}: {v}")

print("\nЕсли val_f1 < 0.85 после 3 эпох:")
print("  1. Попробуй r=16, alpha=32")
print("  2. Добавь 'key' в target_modules")
print("  3. Если всё равно плохо — переходи на finetuning_type: full")

=== Рекомендуемая конфигурация для спам-классификации ===

configs/model/builder/default.yaml:
  peft_config:
    r: 8
    lora_alpha: 16
    target_modules: ['query', 'value']
    lora_dropout: 0.05
    bias: none

Если val_f1 < 0.85 после 3 эпох:
  1. Попробуй r=16, alpha=32
  2. Добавь 'key' в target_modules
  3. Если всё равно плохо — переходи на finetuning_type: full
